> 📄 English: [search-extract-crawl.ipynb](search-extract-crawl.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl)

# 1. 搜索、提取与抓取网络 🌐

欢迎！在本教程中，你将亲身实践 Tavily API 的核心能力——以语义理解搜索网络、从实时网页提取内容，以及抓取整个网站。

对于任何需要来自互联网的最新、相关信息的 AI Agent 或应用程序构建者而言，这些技能都必不可少。通过学习如何以编程方式访问和处理实时 Web 数据，你将能够弥合静态语言模型与其所处的动态世界之间的鸿沟，让你的 Agent 更聪明、更准确、更贴合上下文。

我们将涵盖：
- 如何执行 Web 搜索并检索最相关的结果
- 如何从任意 URL 提取干净、可用的内容
- 如何抓取网站以收集全面信息
- 如何使用高级参数微调你的查询


---


## 快速开始

请按照以下步骤配置：

1. **注册** Tavily：访问 [app.tavily.com](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-app&target=https%3A%2F%2Fapp.tavily.com%2Fhome%2F%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=app.tavily.com) 获取你的 API key。

   *可参考下方链接的截图，获取分步指引：*

   - ![截图：注册页面](assets/sign-up.png)
   - ![截图：Tavily API Keys 控制台](assets/api-key.png)


2. 从你的 Tavily 账号控制台**复制你的 API key**。

3. 将你的 API key**粘贴**到下面的单元格中并执行该单元格。


In [ ]:
# To export your API key into a .env file, run the following cell (replace with your actual keys):
!echo "TAVILY_API_KEY=<your-tavily-api-key>" >> .env

在下面的单元格中安装依赖。

In [ ]:
%pip install --upgrade tavily-python --quiet

### 配置你的 Tavily API 客户端

下面的代码将使用你的 API key 实例化 Tavily 客户端。


In [ ]:
import os
import getpass
from dotenv import load_dotenv
from tavily import TavilyClient

# Load environment variables from .env file
load_dotenv()

# Prompt the user to securely input the API key if not already set in the environment
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

# Initialize the Tavily API client using the loaded or provided API key
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

## 搜索 Search 🔍

让我们运行一个基础的 Web 搜索查询，检索关于纽约的最新信息。


In [ ]:
# Perform a web search to retrieve the most up-to-date information available.
search_results = tavily_client.search(
    query="What happened in NYC today?", max_results=5
)

这次搜索调用将返回 5 个结果。每个结果都包含网页的标题、URL、用于 RAG 的内容片段，以及一个语义得分，表示该结果与你的查询的匹配程度。

In [ ]:
# Print the results
for result in search_results["results"]:
    print(result["title"])
    print(result["url"])
    print(result["content"])
    print(result["score"])
    print("\n")

让我们试验不同的 API 参数配置，看看 Tavily 的实际表现。从宽泛的主题到具体的问题都可以尝试！你可以调整结果数量、时间范围和域名过滤等参数来定制搜索。更多信息请阅读搜索 [API 参考](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-docs&target=https%3A%2F%2Fdocs.tavily.com%2Fdocumentation%2Fapi-reference%2Fendpoint%2Fsearch%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=API%20reference)和[最佳实践指南](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-docs&target=https%3A%2F%2Fdocs.tavily.com%2Fdocumentation%2Fbest-practices%2Fbest-practices-search%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=best%20practices%20guide)。让我们应用时间范围过滤、域名过滤，并使用 `news` 搜索主题。


In [ ]:
# Perform a web search with a specific time range, domain filter, and topic filter.
search_results = tavily_client.search(
    query="Anthropic model release?",
    max_results=5,
    time_range="month",
    include_domains=["techcrunch.com"],
    topic="news",
)

注意，所有结果都来自 `techcrunch.com`，且仅限于过去一个月。通过设置 `news` 主题，我们的搜索聚焦于可信的第三方新闻来源。

In [ ]:
# Print the results
for result in search_results["results"]:
    print(result["title"])
    print(result["url"])
    print(result["content"])
    print("\n")

## 提取 Extract 📄

接下来，我们将使用 Tavily 的 extract 端点，通过之前搜索结果中的 URL 检索每个页面的完整内容（即 `raw_content`）。相比仅使用搜索中的简短内容片段，这让我们能够访问每个页面的完整文本。为提高效率，extract 端点可以在单次调用中同时处理最多 20 个 URL。更多信息请阅读 extract [API 参考](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-docs&target=https%3A%2F%2Fdocs.tavily.com%2Fdocumentation%2Fapi-reference%2Fendpoint%2Fextract%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=API%20reference)和[最佳实践指南](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-docs&target=https%3A%2F%2Fdocs.tavily.com%2Fdocumentation%2Fbest-practices%2Fbest-practices-extract%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=best%20practices%20guide)。


In [ ]:
# Extract the full-page content from the URLs in the search results.
extract_results = tavily_client.extract(
    urls=[result["url"] for result in search_results["results"]],
    # extract_depth="advanced", #uncomment to use our advanced extract depth for complex web pages with dynamic content, embedded media, or structured data.
)

让我们看看原始内容，它比之前显示的简短内容片段提供了更详细、更完整的信息。如果你将原始内容作为输入提供给 LLM，请记得考虑模型上下文窗口的限制。

In [ ]:
# Print the results
for result in extract_results["results"]:
    print(result["url"])
    print(result["raw_content"])
    print("\n")

与其使用 Extract 端点返回原始页面内容，我们可以通过使用带 `include_raw_content=True` 参数的 search 端点，将搜索与提取合并为一次 API 调用。

In [ ]:
# Perform a web search with live content extraction.
search_results = tavily_client.search(
    query="Anthropic model release?",
    max_results=1,
    include_raw_content=True,
)

每个搜索结果现在都包含网页的标题、URL、语义得分、内容片段以及完整的原始内容。Tavily 灵活而模块化的 API 支持构建各类 Agent 系统，无论模型规模大小。

In [ ]:
# Print the results
for result in search_results["results"]:
    print(result["url"])
    print(result["content"])
    print(result["score"])
    print(result["raw_content"])
    print("\n")

## 抓取 Crawl 🕸️

现在让我们使用 Tavily 抓取一个网页并提取其所有链接。Web 抓取（crawling）是通过跟随超链接自动浏览网站以发现大量网页和 URL 的过程（可以想象成掉进维基百科的兔子洞 🐇——从一个页面点到另一个页面，深入相互关联的主题）。对于自主 Web Agent 而言，这种能力对于访问可能难以通过搜索获取的深层 Web 数据至关重要。更多信息请阅读 crawl [API 参考](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-docs&target=https%3A%2F%2Fdocs.tavily.com%2Fdocumentation%2Fapi-reference%2Fendpoint%2Fcrawl%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=API%20reference)和[最佳实践指南](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-docs&target=https%3A%2F%2Fdocs.tavily.com%2Fdocumentation%2Fbest-practices%2Fbest-practices-crawl%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=best%20practices%20guide)。


让我们从抓取 Tavily 网站开始，收集所有嵌套页面。

In [ ]:
# Crawl the Tavily website
crawl_results = tavily_client.crawl(url="tavily.com")

我们可以看到所有嵌套的 URL。

In [ ]:
# Print the results
for result in crawl_results["results"]:
    print(result["url"])

crawl 端点也会返回每个 URL 的原始页面内容。

In [ ]:
# Print the results
for result in crawl_results["results"]:
    print(result["url"])
    print(result["raw_content"])
    print("\n")

如果你只对链接感兴趣（而不需要完整的页面内容），请使用 Map 端点。它是一种更快、更经济高效的方式来检索站点上的所有链接。

In [ ]:
# Map the Tavily website
map_results = tavily_client.map(url="tavily.com")

让我们查看结果，在这种情况下它们只包含链接。

In [ ]:
# Print the results
map_results

crawl/map 端点的 `instructions` 参数是一个强大的功能，让你可以用自然语言指令来引导网络抓取。

In [ ]:
# Map the Tavily website with natural language instructions
guided_map_results = tavily_client.map(
    url="tavily.com", instructions="find only the developer docs"
)

现在，结果将只包含来自 Tavily 网页的开发者文档。

In [ ]:
guided_map_results

尝试不同的 URL，看看 Tavily 是如何映射不同网站结构的。你会如何将它集成到你的 Agent 系统中……🤔？

## 结论与后续步骤

在本教程中，你学会了如何：
- 使用 Tavily API 执行实时 Web 搜索
- 从网页提取内容
- 抓取和映射网站以收集链接和信息
- 用自然语言指令引导抓取，实现有针对性的数据提取

这些基础技能让你的 Agent 能够访问并利用最新的 Web 信息，使其更强大、更贴合上下文。你可以在 [playground](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-app&target=https%3A%2F%2Fapp.tavily.com%2Fplayground%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=playground) 中自由试验 Tavily API，并阅读[最佳实践指南](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--search-extract-crawl&click=tavily-docs&target=https%3A%2F%2Fdocs.tavily.com%2Fdocumentation%2Fbest-practices%2Fbest-practices-search%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=best%20practices%20guide) 以针对你的用例进行优化。

**准备好进行下一步了吗？**
在**教程 #2：构建 Web Agent**中，你可以自主搜索、提取和抓取，你将组合这些能力构建一个完全自主的 Web Agent。这个 Agent 将能够推理、决定何时搜索、抓取或提取，并将 Web 数据集成到其工作流中——全部由 Tavily 驱动。

[👉 **继续到教程 #2！**](./web-agent-tutorial.ipynb)
